<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_2_model_xgboost.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_2_model_xgboost

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Clonado de repositorio / Acceso a Drive

In [1]:
#Clonamos el repo
#LINK DE REPOSITORIO: https://github.com/GUNAPILLCO/neural_profit
!git clone https://github.com/GUNAPILLCO/neural_profit.git

fatal: destination path 'neural_profit' already exists and is not an empty directory.


In [2]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 0.2. Instalación de librerías


In [3]:
#!{sys.executable} -m pip install -q ta
#print("Librería instalada: technical-analysis")

In [4]:
!pip install --no-cache-dir "xgboost==2.1.1" -q

!pip uninstall -y scikit-learn scikit-image threadpoolctl joblib --quiet
!pip install --no-cache-dir "scikit-learn==1.5.1" "optuna==3.6.1" "numpy>=1.24,<3" "scipy>=1.10" --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 274.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.4/308.4 kB 388.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.1 which is incompatible.


In [5]:
!pip install optuna --quiet

### 0.3. Importación de librerías


In [6]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import sklearn, scipy
import sklearn, numpy, scipy, optuna
from sklearn.ensemble import RandomForestRegressor

import joblib
import optuna
from tqdm import tqdm

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import xgboost as xgb
from xgboost import XGBRegressor

In [7]:
print("python:", sys.version)
print("sklearn:", sklearn.__version__)
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("optuna:", optuna.__version__)
print("xgboost:", xgb.__version__)


python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
sklearn: 1.5.1
numpy: 2.0.2
scipy: 1.16.2
optuna: 3.6.1
xgboost: 2.1.1


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [8]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [9]:
mnq_train = load_data("train")
mnq_valid = load_data("valid")
mnq_test = load_data("test")

### 1.2. Información de datasets


In [10]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [11]:
info_dataset(mnq_train, 'mnq_train')
info_dataset(mnq_valid, 'mnq_valid')
info_dataset(mnq_test, 'mnq_test')

Información del dataset mnq_train:

	Cantidad de días: 917
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_valid:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York

Información del dataset mnq_test:

	Cantidad de días: 197
	Registros por día: 301
	Hora diaria de inicio 09:30
	Hora diaria de final 14:30
	Zona horaria: America/New_York



(197, np.float64(301.0))

### 1.3. Carga de listado de features por ventana de tiempo

In [12]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_30 = features_dict["features_to_30"]
features_to_60 = features_dict["features_to_60"]
features_to_90 = features_dict["features_to_90"]


In [13]:
print(f'Listado de features para 30min: {features_to_30}')
print(f'Listado de features para 60min: {features_to_60}')
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 30min: ['ire_90', 'rev_mom_z_90', 'roc_60', 'rev_score_90', 'price_ema30']
Listado de features para 60min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'rev_mom_vol_z_60', 'momentum_5', 'roc_20']
Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [14]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [15]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [16]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [17]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [18]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [19]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [20]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [21]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

In [22]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [23]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [24]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [25]:
xgb_metrics, metrics = load_or_create_metrics("4_2_xgboost_metrics")

Las métricas no existen. Se crea el dataset xgboost_metrics para almacenar las métricas


In [49]:
def save_metrics (metrics,  metrics_name: str):   #("4_2_xgboost_metrics")
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

In [26]:
xgb_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


In [27]:
import xgboost as xgb
print("XGBoost compiled with CUDA:", xgb.__version__ >= "1.6.0")  # GPU build check

XGBoost compiled with CUDA: True


## 4. Entrenamiento de Modelo

### 4.0. Funciones

#### Función para evaluación de modelo


In [28]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [29]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

#### Función para subsamplear

In [30]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

In [31]:
xgb_device_test = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")
try:
    xgb_device_test.fit([[0,0],[1,1]], [0,1])
    print("✅ XGBoost GPU works correctly")
except Exception as e:
    print("❌ GPU not available for XGBoost:", e)

✅ XGBoost GPU works correctly


#### Función para entrenamiento

In [32]:
def train_model (best_params, X_train, y_train, X_valid, y_valid):

    # Entrenamos el model final con los mejores hiperparámetros
    # Se entrena un RandomForest definitivo usando todos los datos de entrenamiento y los best_params
    model = XGBRegressor(**best_params, n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)

    # Evaluación en validación
    preds = model.predict(X_valid)

    return model, preds

#### Parámetros por defecto

In [33]:
xgb_default_params = {
    "n_estimators": 800,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_lambda": 1.0,
    "reg_alpha": 0.0,
    #"random_state": 42,
    #"n_jobs": -1,
    "tree_method": "gpu_hist",         # CPU rápido; cambiar a "gpu_hist" si tenés GPU CUDA disponible
    "objective": "reg:squarederror"
}

### 4.1. Entrenamiento 30min

In [34]:
def run_xgb_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled,
    y_train,
    X_valid_scaled,
    y_valid,
    resample_rate: float,                  # 0.3, 0.5, 1.0
    xgb_params: dict,
    xgb_metrics_df: pd.DataFrame | None = None,
    n_samples_train: int | None = None,
    n_samples_valid: int | None = None,
):
    """
    Ejecuta un experimento XGBoost con (opcional) subsampleo de train/valid.

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en xgb_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'XGB_30_subsampleado_30%').
    X_train_scaled, y_train : arrays
        Ventanas y target de entrenamiento (escalados ya).
    X_valid_scaled, y_valid : arrays
        Ventanas y target de validación (escalados ya).
    resample_rate : float
        Proporción a muestrear (0 < r <= 1). 1.0 = sin subsampleo.
    xgb_params : dict
        Diccionario de hiperparámetros para XGBRegressor (p.ej. xgb_default_params).
    xgb_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.
    n_samples_train, n_samples_valid : int | None
        Tamaños base para calcular la cantidad a muestrear. Si es None, se infiere de X_*.

    Retorna
    -------
    dict
        Diccionario con métricas {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """
    # Si ya hay métricas guardadas y metrics_flag=True, sólo mostrarlas y devolverlas
    if metrics_flag is True and xgb_metrics_df is not None and model_key in xgb_metrics_df.index:
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = xgb_metrics_df.loc[model_key].to_dict()
        print_metrics(metrics_dict, model_key)
        return metrics_dict

    # Mensaje de entrenamiento
    tag_rate = f"{int(resample_rate*100)}%" if resample_rate < 1.0 else "100%"
    print(f"Entrenando modelo {model_key} (resample={tag_rate})...")

    # Calcular tamaños a muestrear (si no los pasan, inferir de X)
    if n_samples_train is None:
        n_samples_train = X_train_scaled.shape[0]
    if n_samples_valid is None:
        n_samples_valid = X_valid_scaled.shape[0]

    # Subsampleo (si corresponde)
    if resample_rate < 1.0:
        n_train_sub  = max(1, int(n_samples_train * resample_rate))
        n_valid_sub  = max(1, int(n_samples_valid * resample_rate))
        X_train_sub, y_train_sub = subsample(X_train_scaled, y_train, n_train_sub)
        X_valid_sub, y_valid_sub = subsample(X_valid_scaled, y_valid, n_valid_sub)
    else:
        X_train_sub, y_train_sub = X_train_scaled, y_train
        X_valid_sub, y_valid_sub = X_valid_scaled, y_valid

    # Entrenamiento y predicción en valid
    model, y_pred_valid = train_model(
        xgb_params,
        X_train_sub, y_train_sub,
        X_valid_sub, y_valid_sub
    )

    # Evaluación
    metrics_dict = evaluate_model(model, X_valid_sub, y_valid_sub, y_pred=y_pred_valid)

    # Mostrar
    print_metrics(metrics_dict, model_key)

    # Persistir métricas en el DataFrame si se pasa
    if xgb_metrics_df is not None:
        xgb_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict

#### 4.1.1. Con 30% de dataset

In [35]:
xgb_30_30 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_30,
    n_samples_valid=n_samples_valid_30
)

Entrenando modelo XGB_30_subsampleado_30% (resample=30%)...
Métricas de XGB_30_subsampleado_30%:

	 RMSE:	 0.002928
	  MAE:	 0.001628
	   R2:	 0.278324
	SMAPE:	 114.270053
	DirAcc:	 0.712751


#### 4.1.2. Con 50% de dataset

In [36]:
xgb_30_50 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_30,
    n_samples_valid=n_samples_valid_30
)

Entrenando modelo XGB_30_subsampleado_50% (resample=50%)...
Métricas de XGB_30_subsampleado_50%:

	 RMSE:	 0.002702
	  MAE:	 0.001599
	   R2:	 0.292092
	SMAPE:	 114.573267
	DirAcc:	 0.712746


#### 4.1.3. Con ventanas completas

In [37]:
xgb_30_100 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_30_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1.0,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_30,
    n_samples_valid=n_samples_valid_30
)

Entrenando modelo XGB_30_100% (resample=100%)...
Métricas de XGB_30_100%:

	 RMSE:	 0.002774
	  MAE:	 0.001606
	   R2:	 0.281533
	SMAPE:	 114.170563
	DirAcc:	 0.711309


In [38]:
xgb_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
XGB_30_subsampleado_30%,0.002928,0.001628,0.278324,114.270053,0.712751
XGB_30_subsampleado_50%,0.002702,0.001599,0.292092,114.573267,0.712746
XGB_30_100%,0.002774,0.001606,0.281533,114.170563,0.711309


### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [41]:
xgb_60_30 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_60,
    n_samples_valid=n_samples_valid_60
)


Entrenando modelo XGB_60_subsampleado_30% (resample=30%)...
Métricas de XGB_60_subsampleado_30%:

	 RMSE:	 0.003788
	  MAE:	 0.001965
	   R2:	 0.376420
	SMAPE:	 96.786526
	DirAcc:	 0.785565


#### 4.2.2. Con 50% de dataset

In [42]:
xgb_60_50 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_60,
    n_samples_valid=n_samples_valid_60
)

Entrenando modelo XGB_60_subsampleado_50% (resample=50%)...
Métricas de XGB_60_subsampleado_50%:

	 RMSE:	 0.003686
	  MAE:	 0.001936
	   R2:	 0.407545
	SMAPE:	 95.167915
	DirAcc:	 0.790887


#### 4.2.3. Con ventanas completas

In [43]:
xgb_60_100 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_60_100%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=1.0,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_60,
    n_samples_valid=n_samples_valid_60
)

Entrenando modelo XGB_60_100% (resample=100%)...
Métricas de XGB_60_100%:

	 RMSE:	 0.003547
	  MAE:	 0.001919
	   R2:	 0.416467
	SMAPE:	 95.899622
	DirAcc:	 0.786994


In [44]:
xgb_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
XGB_30_subsampleado_30%,0.002928,0.001628,0.278324,114.270053,0.712751
XGB_30_subsampleado_50%,0.002702,0.001599,0.292092,114.573267,0.712746
XGB_30_100%,0.002774,0.001606,0.281533,114.170563,0.711309
XGB_60_subsampleado_30%,0.003788,0.001965,0.376420,96.786526,0.785565
XGB_60_subsampleado_50%,0.003686,0.001936,0.407545,95.167915,0.790887
XGB_60_100%,0.003547,0.001919,0.416467,95.899622,0.786994


### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [45]:
xgb_90_30 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_90,
   n_samples_valid=n_samples_valid_90,
)

Entrenando modelo XGB_90_subsampleado_30% (resample=30%)...
Métricas de XGB_90_subsampleado_30%:

	 RMSE:	 0.004381
	  MAE:	 0.002221
	   R2:	 0.458035
	SMAPE:	 88.668659
	DirAcc:	 0.814755


#### 4.3.2. Con 50% de dataset

In [46]:
xgb_90_50 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_90,
    n_samples_valid=n_samples_valid_90
)

Entrenando modelo XGB_90_subsampleado_50% (resample=50%)...
Métricas de XGB_90_subsampleado_50%:

	 RMSE:	 0.004447
	  MAE:	 0.002197
	   R2:	 0.440817
	SMAPE:	 87.895017
	DirAcc:	 0.816340


#### 4.3.3. Con ventanas completas

In [47]:
xgb_90_100 = run_xgb_experiment(
    metrics_flag=False,
    model_key="XGB_90_100%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=1.0,
    xgb_params=xgb_default_params,
    xgb_metrics_df=xgb_metrics,
    n_samples_train=n_samples_train_90,
    n_samples_valid=n_samples_valid_90
)

Entrenando modelo XGB_90_100% (resample=100%)...
Métricas de XGB_90_100%:

	 RMSE:	 0.004334
	  MAE:	 0.002167
	   R2:	 0.457986
	SMAPE:	 87.189119
	DirAcc:	 0.818053


## 5. Recuperación de métricas

In [70]:
xgb_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
XGB_30_subsampleado_30%,0.002928,0.001628,0.278324,114.270053,0.712751
XGB_30_subsampleado_50%,0.002702,0.001599,0.292092,114.573267,0.712746
XGB_30_100%,0.002774,0.001606,0.281533,114.170563,0.711309
XGB_60_subsampleado_30%,0.003788,0.001965,0.376420,96.786526,0.785565
XGB_60_subsampleado_50%,0.003686,0.001936,0.407545,95.167915,0.790887
XGB_60_100%,0.003547,0.001919,0.416467,95.899622,0.786994
XGB_90_subsampleado_30%,0.004381,0.002221,0.458035,88.668659,0.814755
XGB_90_subsampleado_50%,0.004447,0.002197,0.440817,87.895017,0.816340
XGB_90_100%,0.004334,0.002167,0.457986,87.189119,0.818053


Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [67]:
def generate_metrics_xgb():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  xgb_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
  xgb_30_30 =   {
      'RMSE': 0.002928098505693225,
      'MAE': 0.0016282318127160585,
      'R2': 0.2783242111188996,
      'SMAPE': 114.2700529938078,
      'DirAcc': 0.7127506014434644
  }

  xgb_30_50 = {
      'RMSE': 0.0027018115133004557,
      'MAE': 0.0015993206963948242,
      'R2': 0.29209197495841166,
      'SMAPE': 114.57326664863284,
      'DirAcc': 0.7127459943222826
  }

  xgb_30_100 = {
      'RMSE': 0.002773854508997797,
      'MAE': 0.0016059645806583043,
      'R2': 0.2815330040890046,
      'SMAPE': 114.17056349575311,
      'DirAcc': 0.7113094522096856
  }


  xgb_60_30 = {
      'RMSE': 0.0037881500813111515,
      'MAE': 0.0019650505709726987,
      'R2': 0.3764198093942468,
      'SMAPE': 96.78652621527985,
      'DirAcc': 0.7855653568564555
  }

  xgb_60_50 = {
      'RMSE': 0.0036857665171224473,
      'MAE': 0.001935639758221075,
      'R2': 0.40754502220976474,
      'SMAPE': 95.16791457034456,
      'DirAcc': 0.790886782466439
  }

  xgb_60_100 = {
      'RMSE': 0.003546948576261278,
      'MAE': 0.0019186126576007248,
      'R2': 0.416466784021834,
      'SMAPE': 95.89962163430053,
      'DirAcc': 0.7869944908220463
  }


  xgb_90_30 = {
      'RMSE': 0.004380854048112429,
      'MAE': 0.002220670339021563,
      'R2': 0.4580347171389979,
      'SMAPE': 88.6686592119478,
      'DirAcc': 0.8147554129911788
  }

  xgb_90_50 = {
      'RMSE': 0.004447452015457231,
      'MAE': 0.002196836270087189,
      'R2': 0.44081652850192843,
      'SMAPE': 87.89501721981583,
      'DirAcc': 0.8163402781119184
  }

  xgb_90_100 = {
    'RMSE': 0.004334062373094705,
    'MAE': 0.0021670116276276638,
    'R2': 0.45798627347485654,
    'SMAPE': 87.18911892470159,
    'DirAcc': 0.8180527822551543
  }

  xgb_metrics.loc['XGB_30_subsampleado_30%'] = xgb_30_30
  xgb_metrics.loc['XGB_30_subsampleado_50%'] = xgb_30_50
  xgb_metrics.loc['XGB_30_100%'] = xgb_30_100

  xgb_metrics.loc['XGB_60_subsampleado_30%'] = xgb_60_30
  xgb_metrics.loc['XGB_60_subsampleado_50%'] = xgb_60_50
  xgb_metrics.loc['XGB_60_100%'] = xgb_60_100

  xgb_metrics.loc['XGB_90_subsampleado_30%'] = xgb_90_30
  xgb_metrics.loc['XGB_90_subsampleado_50%'] = xgb_90_50
  xgb_metrics.loc['XGB_90_100%'] = xgb_90_100

  save_metrics(xgb_metrics, "4_2_xgboost_metrics")
  return xgb_metrics

In [ ]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#generate_metrics_xgb()

In [69]:
save_metrics(xgb_metrics, "4_2_xgboost_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_2_xgboost_metrics.parquet
